# MobileNetV4-Small — Model Analysis
**Member 1 deliverable.** Loads the pretrained model, modifies the classification head for CIFAR-100, and analyzes parameters / MACs / forward-pass correctness.

## 1. Setup

In [ ]:
!nvidia-smi
!pip install -U timm thop


In [ ]:
import torch
import timm

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Load MobileNetV4-Small (pretrained)

model = timm.create_model('mobilenetv4_conv_small.e2400_r224_in1k', pretrained=True)
model.eval();

num_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model loaded — {num_params:.2f}M parameters (ImageNet head, 1000 classes)")

In [ ]:
# 3. Modify classification head for CIFAR-100 (100 classes)

print("Before:", model.get_classifier())

model.reset_classifier(num_classes=100)

print("After: ", model.get_classifier())

## 4. Inspect architecture — Universal Inverted Bottleneck (UIB) blocks

timm reuses its `MobileNetV3` builder class for MobileNetV4 (hence the class name printed below is `MobileNetV3`), but the actual blocks are `UniversalInvertedResidual` — this is the UIB block from the paper's §4.

Each `UniversalInvertedResidual` block has sub-layers (`dw_start`, `pw_exp`, `dw_mid`, `se`, `pw_proj`, `dw_end`) that can each be either an active conv layer or `Identity()`. Which ones are active determines whether a given block behaves like an ExtraDW, IB (Inverted Bottleneck), or ConvNeXt-like variant from Fig. 3 of the paper:
- **ExtraDW-style**: `dw_start` and `dw_mid` both active (two depthwise convs)
- **IB-style (classic inverted bottleneck)**: `dw_start` is `Identity()`, only `dw_mid` active
- **ConvNeXt-like**: `dw_start` active, `dw_mid` is `Identity()` (single depthwise conv before the pointwise expansion)

In [ ]:
print(model)

## 5. Parameter count (accurate, via PyTorch)

In [ ]:
total_params = sum(p.numel() for p in model.parameters()) / 1e6
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"Total params:     {total_params:.3f} M")
print(f"Trainable params: {trainable_params:.3f} M")

## 6. MACs analysis (via thop) + forward-pass sanity check

**Note:** thop only registers `Conv2d`, `ReLU`, `Sequential`, and `AdaptiveAvgPool2d` layers (see the `[INFO] Register...` lines below) — it silently skips `Linear` and `BatchNorm` layers, so its param count under-reports vs. Section 5 above. Use thop's **MACs** number as the reliable efficiency metric; use Section 5's param count as the accurate total.

In [ ]:
from thop import profile

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval();

dummy_input = torch.randn(1, 3, 224, 224).to(device)

macs, thop_params = profile(model, inputs=(dummy_input,))

print(f"MACs:        {macs / 1e9:.3f} G")
print(f"thop params: {thop_params / 1e6:.3f} M  (undercounts vs. Section 5 — see note above)")

with torch.no_grad():
    output = model(dummy_input)
print("Output shape:", output.shape, "(expected: [1, 100])")

## 7. Summary

| Metric | Value |
|---|---|
| Total parameters | 3.77 M |
| MACs (224x224 input) | 0.185 G |
| Output shape | [1, 100] |
| Expected input size | 224x224 |

Model is ready to hand off — clean loading code lives in `mobilenetv4.py` (`get_model()`), which Member 2 imports for the training pipeline. This notebook stays as the analysis record; it is not part of the training pipeline.

## 8. Data input config (for reference — Member 2's pipeline should match this)

In [ ]:
data_config = timm.data.resolve_model_data_config(model)
print(data_config)